In [2]:
import pandas as pd
import glob
from pathlib import Path
import json

In [3]:
#loading the .jsonl log files
# df_m10 = pd.read_json("/Users/eagmurray/Projects/DataAnalysis/Q40/flight_tests_12_May/noAirRaid/Flight_1_noAR_100mAlt/decoded_log.jsonl", lines=True)
df_q40 = pd.read_json("/Users/eagmurray/Projects/DataAnalysis/Q40/flight_tests_12_May/noAirRaid/Flight_2_noAR_30mAlt/q40_nav_raw_log.jsonl",lines=True)

In [4]:
def append_jsonl(path: Path, record: dict):
    """
    Records dictionary to a log file from the decoded ASN.1 stream
    
    """
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, separators=(",", ":"), default=str))
        f.write("\n")

def swap_bytes_in_words(data: bytes) -> bytes:
    """
    This function is called to swap the message payload in big-endian encoding, as asn1tools expects big-endian encoding
    """
    if len(data) % 2 != 0:
        raise ValueError("Payload length must be even")
    out = bytearray()
    for i in range(0, len(data), 2):
        out.append(data[i + 1])
        out.append(data[i])
    return bytes(out)

In [5]:
import asn1tools
asn1_files = glob.glob("/Users/eagmurray/Projects/DataAnalysis/Q40/flight_tests_12_May/ASN.1_files/*.asn")
schema = asn1tools.compile_files(asn1_files, "uper")

qflag_values = Path("qflag_values.jsonl")

In [6]:
print(qflag_values)
for i in range(len(df_q40["raw"])):
    payload = bytes.fromhex(df_q40["raw"][i])
    decoded = schema.decode("DataCoreNavSolution", payload)
    
    append_jsonl(qflag_values, {"qflag": decoded["qFlag"]})

qflag_values.jsonl
